# Axis 1 — Cache Capacity vs Working Set (Mac / MLX)

**Question.** At what cache capacity **K** does the working set saturate, and when does adding memory stop paying off?

**Hypothesis (from the team page).** At small caches (*K < 8*) the system is purely I/O-bound. Beyond a knee near *K ≈ 24* the hit rate plateaus because tail experts are rarely activated, and the bottleneck shifts from storage bandwidth to GPU compute.

> **Units.** OLMoE routes every token to 8 experts *per layer*, so "K < 8" and "K ≈ 24" only make sense **per layer**. This notebook sweeps the **global** K over all 1,024 logical `(layer, expert_id)` experts and reports the per-layer equivalent K/16. The hypothesis points are therefore global **K = 128** (8/layer) and **K = 384** (24/layer).

## What is measured
| Metric | How |
|---|---|
| Hit rate (overall, prefill, decode) | real cache counters; extended to every K by exact trace replay |
| Decode throughput, TTFT | wall clock, every step synchronised |
| I/O vs non-I/O time | time inside disk reads vs everything else |
| Working set W(τ) | distinct experts touched in τ consecutive generated tokens |
| Memory | bank bytes (exactly K × 12.6 MB), MLX active/peak, process RSS |
| Cold vs steady state | first request (empty cache) vs the rest |

## The cache is real
Only the non-expert weights (~0.95 GB) are loaded. The 12.9 GB expert bank stays in the safetensors files. A cache of capacity K is a preallocated bank of K slots; a miss reads that expert's bytes **from disk straight into its slot** (`pread` with `F_NOCACHE`, so the OS page cache does not quietly hold the bank). Expert compute is `mx.gather_mm` over the slot bank. See `experiments/moe_cache.py`.

**Runtime.** The full sweep serves 12 requests × 64 new tokens at each K that fits in RAM: roughly 30–60 minutes on an M4 / 16 GB. Set `MOE_SMOKE=1` in the environment for a two-minute check.

In [ ]:
import os, sys, time, json, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlx.core as mx

# Find the repo and import the shared module (experiments/moe_cache.py).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiments' / 'moe_cache.py').exists())
sys.path.insert(0, str(ROOT / 'experiments'))
import moe_cache as mc
importlib.reload(mc)

# MOE_SMOKE=1 runs a tiny configuration end to end (used to test the notebook).
SMOKE = os.environ.get('MOE_SMOKE') == '1'
AXIS_DIR = ROOT / 'experiments' / 'axis-1-cache-capacity'
RESULTS = (ROOT / 'experiments' / '.cache' / 'smoke' / 'axis-1-cache-capacity') if SMOKE else (AXIS_DIR / 'results')
RESULTS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.figsize': (11, 5), 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})
print('SMOKE run' if SMOKE else 'Full run', '| results ->', RESULTS.relative_to(ROOT))
print(mc.system_info())

## Configuration

In [ ]:
K_VALUES = [64, 96, 128, 192, 256, 384, 512, 768, 1024]   # global K (per-layer = K/16)
POLICY = 'LRU'                 # Axis 1 holds the policy fixed; Axis 2 varies it
PROMPTS_PER_DOMAIN = 3         # x 4 kinds = 12 requests served back to back (max 4)
MAX_NEW_TOKENS = 64            # fixed length: EOS does not stop generation
PURGE_BEFORE_EACH_K = True     # drop the OS file cache first (needs passwordless `sudo purge`)

if SMOKE:
    K_VALUES, PROMPTS_PER_DOMAIN, MAX_NEW_TOKENS = [64, 256], 1, 8

print('Global K   :', K_VALUES)
print('Per-layer  :', [k / 16 for k in K_VALUES])

## Model (non-expert weights only) and correctness gate

In [ ]:
model, tokenizer, index = mc.load_model()
print(index.describe())
print(f'Non-expert weights resident: {mc.non_expert_bytes(model) / 1e9:.2f} GB '
      f'| MLX active memory: {mx.get_active_memory() / 1e9:.2f} GB')
print(f'Largest K that fits safely in this machine\'s RAM: {mc.max_safe_capacity(index)}')

# Correctness gate: one layer through the K-slot bank vs the stock mlx-lm layer
# (all 64 experts of that layer stacked). Must be bit-identical.
diff = mc.check_layer_exactness(model, index, capacity=64)
assert diff == 0.0, diff
print(f'Layer exactness vs stock mlx-lm: max |diff| = {diff}')

## Workload
Axis 1 asks how big the working set of *ordinary serving traffic* is, so its workload is a mix of four kinds of request (math word problems, coding tasks, everyday chat, writing), **interleaved** so the cache sees mixed traffic as a server would. The prompts are written for this axis and live in the next cell; the same requests are served at every K.

In [ ]:
# Mixed serving traffic: four kinds of request, interleaved, as a chat server would see.
PROMPTS = {
    'math': [
        'A train leaves at 9:40 and arrives at 13:15. If it travels 287 km, what is its average speed? Show your steps.',
        'A shop sells pens at 3 for $2. How much do 45 pens cost, and how many pens can you buy with $30?',
        'The sum of three consecutive even numbers is 138. Find the numbers and explain your reasoning.',
        'A rectangle has a perimeter of 54 cm and its length is twice its width. What is its area?',
    ],
    'code': [
        'Write a Python function that returns the k most frequent words in a text, ignoring case and punctuation.',
        'Implement an LRU cache class in Python with get and put methods that run in O(1) time.',
        'Write a SQL query that finds the second highest salary in each department of an employees table.',
        'Explain what this does and fix the bug: def avg(xs): return sum(xs) / len(xs) if xs else sum(xs) / 0',
    ],
    'chat': [
        'What are some good ways to stay focused while studying for long hours?',
        'Can you explain the difference between weather and climate in simple terms?',
        'I am visiting Japan for a week in spring. What should I plan for?',
        'How does compound interest work, and why do people say to start saving early?',
    ],
    'writing': [
        'Write a short story about a lighthouse keeper who receives a letter from the future.',
        'Write a persuasive paragraph arguing that cities should plant more trees.',
        'Describe a busy night market using all five senses.',
        'Write a polite email to a professor asking for an extension on an assignment.',
    ],
}
workload = mc.requests(PROMPTS, n_per_domain=PROMPTS_PER_DOMAIN, order='interleave')
wl = pd.DataFrame({'domain': [p['domain'] for p in workload],
                   'prompt_tokens': [len(mc.chat_tokens(tokenizer, p['text'])) for p in workload],
                   'prompt': [p['text'][:70] for p in workload]})
print(f'{len(workload)} requests, {wl.prompt_tokens.sum()} prompt tokens, '
      f'{len(workload) * MAX_NEW_TOKENS} generated tokens per K')
wl

## Real runs
For every K that fits in this machine's RAM: purge the OS file cache, allocate a fresh K-slot bank, serve the whole workload, and record every MoE call. K values that do not fit (the bank alone would exceed ~70% of RAM) are **not timed**; their hit rates come from the exact replay below.

In [ ]:
SAFE_K = mc.max_safe_capacity(index)
real_K = [k for k in K_VALUES if k <= SAFE_K]
replay_only_K = [k for k in K_VALUES if k > SAFE_K]
print(f'Timed (real) runs: {real_K}')
print(f'Replay only (bank would not fit in RAM): {replay_only_K}')

runs, per_prompt, traces, occupancy = {}, [], {}, {}
for K in real_K:
    purged = mc.purge_page_cache() if PURGE_BEFORE_EACH_K else False
    mc.free_memory(); mx.reset_peak_memory()
    cache = mc.ExpertCache(index, K, POLICY)
    print(f'\nK = {K} (per layer {K / 16:g}) | bank {cache.bank_bytes / 1e9:.2f} GB | '
          f'page cache purged: {purged}')
    t0 = time.perf_counter()
    df, rec = mc.run_workload(model, tokenizer, cache, workload, MAX_NEW_TOKENS)
    wall = time.perf_counter() - t0

    s = cache.summary()
    assert s['resident'] <= K and len(cache.free) + s['resident'] == K   # never more than K experts
    df['K'] = K
    per_prompt.append(df)
    runs[K] = {**s, **mc.memory_snapshot(), 'wall_s': wall, 'purged': purged}
    traces[K] = rec.events
    occupancy[K] = np.bincount([l for l, _ in cache.slot_of], minlength=index.num_layers)
    cache.close(); del cache; mc.free_memory()

per_prompt = pd.concat(per_prompt, ignore_index=True)

# The cache must never change the model's output.
texts = per_prompt.groupby('K')['text'].apply(tuple)
assert texts.nunique() == 1, 'generated text differs between K values!'
print('\nGenerated text is identical at every K (the cache is exact).')

## Exact replay: hit rate at every K
Routing does not depend on the cache (the outputs are identical), so the recorded access stream is the same at every K. Replaying it through the same LRU policy, with the same grouping and pinning rules, must reproduce the real cache's counters **exactly** — checked below for every timed K — and then gives the hit rate for K values too large to time.

In [ ]:
events = traces[real_K[0]]
acc = mc.events_frame(events)
print(f'{len(acc):,} cache accesses | {acc[["layer", "eid"]].drop_duplicates().shape[0]} distinct experts touched')

for K in real_K:                       # replay == real, exactly
    pol = mc.make_policy(POLICY, K)
    hits = mc.replay(events, pol)
    assert int(hits.sum()) == runs[K]['hits'] and pol.evictions == runs[K]['evictions'], K
print('Replay reproduces the real cache exactly at K =', real_K)

hit_rows = []
for K in K_VALUES:
    hits = mc.replay(events, mc.make_policy(POLICY, K))
    for phase in ['all', 'prefill', 'decode']:
        m = np.ones(len(acc), bool) if phase == 'all' else (acc.phase == phase).to_numpy()
        hit_rows.append({'K': K, 'phase': phase, 'hit_rate': hits[m].mean()})
hit_curve = pd.DataFrame(hit_rows).pivot(index='K', columns='phase', values='hit_rate')
compulsory = acc[['layer', 'eid']].drop_duplicates().shape[0] / len(acc)
print(f'Compulsory (first-touch) miss share: {compulsory:.1%} -> best possible hit rate {1 - compulsory:.1%}')
hit_curve.style.format('{:.1%}')

## Results table

In [ ]:
agg = per_prompt.groupby('K').agg(
    decode_tokens=('new_tokens', lambda s: (s - 1).sum()), decode_s=('decode_s', 'sum'),
    ttft_mean_s=('ttft_s', 'mean'), io_s=('io_s', 'sum'), wall_s=('wall_s', 'sum'),
    route_wait_s=('route_wait_s', 'sum'), GB_loaded=('GB_loaded', 'sum'))
agg['decode_tok_s'] = agg.decode_tokens / agg.decode_s
agg['io_share'] = agg.io_s / agg.wall_s
agg['bank_GB'] = [runs[k]['bank_GB'] for k in agg.index]
agg['peak_GB'] = [runs[k]['mlx_peak_GB'] for k in agg.index]
agg['ms_per_load'] = [runs[k]['ms_per_load'] for k in agg.index]

table = hit_curve.rename(columns=lambda c: f'hit_{c}').join(agg, how='left')
table.insert(0, 'K_per_layer', table.index / 16)
table.to_csv(RESULTS / 'axis1_summary.csv')
per_prompt.drop(columns='text').to_csv(RESULTS / 'axis1_per_request.csv', index=False)
mc.save_json(RESULTS / 'axis1_runs.json', {str(k): v for k, v in runs.items()})

cols = ['K_per_layer', 'hit_all', 'hit_prefill', 'hit_decode', 'decode_tok_s', 'ttft_mean_s',
        'io_share', 'GB_loaded', 'bank_GB', 'peak_GB', 'ms_per_load']
table[cols].style.format({'K_per_layer': '{:g}', 'hit_all': '{:.1%}', 'hit_prefill': '{:.1%}',
                          'hit_decode': '{:.1%}', 'decode_tok_s': '{:.2f}', 'ttft_mean_s': '{:.2f}',
                          'io_share': '{:.0%}', 'GB_loaded': '{:.1f}', 'bank_GB': '{:.2f}',
                          'peak_GB': '{:.2f}', 'ms_per_load': '{:.2f}'}, na_rep='not timed')

## Hit rate vs K

In [ ]:
knee_K, _ = mc.knee_point(hit_curve.index.values, hit_curve['decode'].values)
fig, ax = plt.subplots()
for phase, style in [('all', '-o'), ('decode', '-s'), ('prefill', '--^')]:
    ax.plot(hit_curve.index, hit_curve[phase], style, label=f'{phase} (replay)')
ax.plot(real_K, [runs[k]['hit_rate'] for k in real_K], 'k*', ms=12, label='timed runs (real cache)')
ax.axhline(1 - compulsory, color='grey', ls=':', label='compulsory-miss ceiling')
for k, txt in [(128, '8 / layer'), (384, '24 / layer')]:
    ax.axvline(k, color='C3', ls='--', alpha=0.5)
    ax.text(k, 0.02, f' hypothesis\n {txt}', color='C3', fontsize=8)
ax.axvline(knee_K, color='C2', lw=2, alpha=0.6, label=f'measured knee (decode): K={knee_K:g}')
ax.set_xscale('log', base=2); ax.set_xticks(K_VALUES); ax.set_xticklabels(K_VALUES)
sec = ax.secondary_xaxis('top', functions=(lambda k: k / 16, lambda k: k * 16))
sec.set_xticks([k / 16 for k in K_VALUES]); sec.set_xticklabels([f'{k / 16:g}' for k in K_VALUES])
sec.set_xlabel('per-layer K')
ax.set_xlabel('global K (resident experts)'); ax.set_ylabel('hit rate'); ax.set_ylim(0, 1)
ax.legend(loc='upper left', fontsize=8); ax.set_title('Hit rate vs cache capacity')
fig.savefig(RESULTS / 'axis1_hit_rate.png', dpi=150, bbox_inches='tight'); plt.show()

## Throughput, latency and where the time goes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
a = agg.loc[real_K]
axes[0].plot(a.index, a.decode_tok_s, '-o'); axes[0].set_title('Decode throughput'); axes[0].set_ylabel('tokens / s')
axes[1].plot(a.index, a.ttft_mean_s, '-o', color='C1'); axes[1].set_title('Mean time to first token'); axes[1].set_ylabel('s')
axes[2].bar([str(k) for k in a.index], a.io_s, label='disk reads (I/O)')
axes[2].bar([str(k) for k in a.index], a.wall_s - a.io_s, bottom=a.io_s, label='everything else (compute + overhead)')
axes[2].set_title('Total time split'); axes[2].set_ylabel('s'); axes[2].legend()
for ax in axes[:2]:
    ax.set_xscale('log', base=2); ax.set_xticks(real_K); ax.set_xticklabels(real_K); ax.set_xlabel('global K')
axes[2].set_xlabel('global K')
fig.tight_layout(); fig.savefig(RESULTS / 'axis1_time.png', dpi=150, bbox_inches='tight'); plt.show()

## Working set W(τ)
How many distinct experts does the model touch in τ consecutive generated tokens? A cache of size K can hold the working set of a window τ when W(τ) ≤ K, so the curve says how much "reuse distance" each K covers.

In [ ]:
taus = [1, 2, 4, 8, 16, 32, 64] if not SMOKE else [1, 2, 4]
ws = mc.working_set_curve(events, taus)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(list(ws), list(ws.values()), '-o', label='W(τ): distinct experts in τ tokens')
for k in K_VALUES:
    ax.axhline(k, color='grey', lw=0.6, alpha=0.5)
    ax.text(taus[-1] * 1.05, k, f'K={k}', fontsize=7, va='center')
ax.axhline(16 * 8, color='C3', ls='--', label='one token needs 8 × 16 = 128')
ax.set_xscale('log', base=2); ax.set_xlabel('τ (generated tokens)'); ax.set_ylabel('distinct (layer, expert)')
ax.legend(); ax.set_title('Decode working set')
fig.savefig(RESULTS / 'axis1_working_set.png', dpi=150, bbox_inches='tight'); plt.show()
pd.Series(ws, name='W(tau)').to_frame().T

## Cold start vs steady state, and how a global cache splits across layers
Request 0 starts with an empty cache; later requests start warm. With a **global** K, layers compete for slots — the heat map shows how many slots each layer holds at the end of the run.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for K in real_K:
    d = per_prompt[per_prompt.K == K]
    axes[0].plot(d.prompt, d.hit_rate, '-o', label=f'K={K}')
axes[0].set_xlabel('request # (0 = cold cache)'); axes[0].set_ylabel('hit rate'); axes[0].legend(fontsize=8)
axes[0].set_title('Per-request hit rate')
occ = np.array([occupancy[k] for k in real_K])
im = axes[1].imshow(occ, aspect='auto', cmap='viridis')
axes[1].set_yticks(range(len(real_K))); axes[1].set_yticklabels([f'K={k}' for k in real_K])
axes[1].set_xlabel('layer'); axes[1].set_title('Resident slots per layer at end of run')
fig.colorbar(im, ax=axes[1]); fig.tight_layout()
fig.savefig(RESULTS / 'axis1_cold_and_layers.png', dpi=150, bbox_inches='tight'); plt.show()

cold = per_prompt[per_prompt.prompt == 0].set_index('K').hit_rate
warm = per_prompt[per_prompt.prompt > 0].groupby('K')[['hits', 'misses']].sum()
steady = warm.hits / (warm.hits + warm.misses)
pd.DataFrame({'cold (request 0)': cold, 'steady (requests 1+)': steady}).style.format('{:.1%}')

## Hypothesis check (descriptive)

In [ ]:
small = [k for k in real_K if k / 16 < 8]
io_share = agg.io_share
cross = [k for k in real_K if io_share[k] < 0.5]
print(f'Measured knee of the decode hit-rate curve: K = {knee_K:g} ({knee_K / 16:g} per layer); '
      f'hypothesis said 384 (24 per layer).')
for k in small:
    print(f'K={k} ({k / 16:g}/layer): I/O is {io_share[k]:.0%} of total time -> '
          f'{"I/O-bound" if io_share[k] > 0.5 else "not I/O-bound"}')
if cross:
    print(f'I/O falls below half of the time from K = {cross[0]} ({cross[0] / 16:g}/layer).')
else:
    print('I/O stays above half of the time at every timed K.')
best = hit_curve['decode'].max()
for frac in (0.9, 0.95):
    kk = hit_curve.index[hit_curve['decode'] >= frac * best][0]
    print(f'{frac:.0%} of the best decode hit rate is reached at K = {kk} ({kk / 16:g}/layer).')

## Notes for reading the results
* **Within-platform only.** These are MLX / F16 / Apple-SSD numbers. Do not compare them directly with the Windows BF16 / PyTorch runs.
* **Page cache.** `F_NOCACHE` stops these reads from filling the OS cache, but data cached *before* the run is still served from RAM. `PURGE_BEFORE_EACH_K` drops it when `sudo purge` works without a password; the `purged` column in `axis1_runs.json` says whether it did.
* **Large K.** At 16 GB of RAM a bank above ~10 GB is not safe, so those K values are replay-only (hit rate exact, no timing).
* **Fixed-length generation.** Every request generates exactly `MAX_NEW_TOKENS`, so work is identical across K.